In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [6]:
def processPdf(path):
    documents = [];
    directory = Path(path)

    pdfFiles = list(directory.glob("**/*.pdf"))

    # print("length: ",len(pdfFiles))

    for pdf in pdfFiles: 
        # print(pdf.name)

        try:
            loader = PyMuPDFLoader(str(pdf));
            doc = loader.load()          
            documents.extend(doc)
        except:
            pass
    return documents

docs = processPdf("../data");
# print(len(docs))

In [ ]:
## test splitting
def createChunks(docs, chunkSize=1000,chunkOverlap=200):
    textSplitter = RecursiveCharacterTextSplitter(
        chunk_size=chunkSize,
        chunk_overlap= chunkOverlap,
        separators=['\n\n', '\n', ' ', ''],
        length_function=len
    )

    splitDocs = textSplitter.split_documents(docs)
    return splitDocs

chunks = createChunks(docs)

In [19]:
# embedding

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
class EmbeddingManager:
    def __init__(self, modelname = "all-MiniLM-L6-v2"):
        self.modelName = modelname
        self.model = None
        self._loadModel()
    
    def _loadModel(self):
        try:
            self.model = SentenceTransformer(self.modelName)
            print("model dimensions",self.model.get_sentence_embedding_dimension())
        except Exception as e:
            print("error: ",e);


    def generateEmbedding(self,text:List[str])->np.ndarray:
        if not self.model:
            print("model not found")
            return;

        print("text length", len(text))
        embedding = self.model.encode(text,show_progress_bar=True)
        print(f"embedding shape {embedding.shape}")
        return embedding;

    def getEmbeddingDimensions(self) -> int:
        if not self.model:
            print("model not loaded");
        return self.model.get_sentence_embedding_dimension()


In [21]:
embedding = EmbeddingManager()
embedding

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7982.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model dimensions 384
